# March Madness 2026 — ML Bracket Predictions

**Authors:** Frank McLaughlin, Jonah Simonson  
**Competition:** UW Madison Sports Analytics Club — March Madness Data Challenge 2026  

The goal is to predict every possible NCAA Tournament matchup (all 2278 of them) and minimize Brier score.

We pulled data from three sources, built 45 features from efficiency stats, shooting splits, and rankings, then trained a tuned Random Forest on 24 years of tournament history. Final CV Brier score: **0.1420** with **81.4% accuracy**.

## Setup

Three data sources:
- `jonathanpilafas/2024-march-madness-statistical-analysis` — KenPom/Barttorvik efficiency stats going back to 2002
- `nishaanamin/march-madness-data` — has the submission template
- Kaggle March Mania 2026 — game results, seeds, regular season results, Massey Ordinals (downloaded via CLI)

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import brier_score_loss
import pickle

dev_path   = kagglehub.dataset_download("jonathanpilafas/2024-march-madness-statistical-analysis")
bart_path  = kagglehub.dataset_download("nishaanamin/march-madness-data")
mania_path = os.path.expanduser('~/Parallels/march_madness/mania_data')

print("Paths loaded")
print(dev_path)
print(mania_path)

## Load Data

Loading everything we need — efficiency stats, game results, seeds, win records, and KenPom rankings.

In [ ]:
# Main efficiency stats dataset (KenPom + Barttorvik combined)
dev = pd.read_csv(os.path.join(dev_path, 'DEV _ March Madness.csv'))
print(f"DEV: {dev.shape} | Years: {sorted(dev['Season'].unique())}")

# March Mania files
teams   = pd.read_csv(os.path.join(mania_path, 'MTeams.csv'))
tourney = pd.read_csv(os.path.join(mania_path, 'MNCAATourneyCompactResults.csv'))
seeds   = pd.read_csv(os.path.join(mania_path, 'MNCAATourneySeeds.csv'))
reg     = pd.read_csv(os.path.join(mania_path, 'MRegularSeasonCompactResults.csv'))
massey  = pd.read_csv(os.path.join(mania_path, 'MMasseyOrdinals.csv'))

# Submission template (all 2278 possible matchups)
matchups = pd.read_csv(os.path.expanduser('~/Parallels/march_madness/2026_Potential_Matchups.csv'))
print(f"Matchups to predict: {len(matchups)}")

# Calculate each team's win% from the regular season
wins   = reg.groupby(['Season','WTeamID']).size().reset_index(name='Wins')
losses = reg.groupby(['Season','LTeamID']).size().reset_index(name='Losses')
record = wins.merge(losses, left_on=['Season','WTeamID'], right_on=['Season','LTeamID'], how='outer')
record['TeamID'] = record['WTeamID'].fillna(record['LTeamID']).astype(int)
record['Wins']   = record['Wins'].fillna(0)
record['Losses'] = record['Losses'].fillna(0)
record['WinPct'] = record['Wins'] / (record['Wins'] + record['Losses'])
record = record[['Season','TeamID','WinPct']]

# Pull KenPom (POM) rankings — grab the latest snapshot before the tournament each year
def grab_rankings(system):
    results = []
    for yr in massey['Season'].unique():
        sub = massey[(massey['SystemName'] == system) & (massey['Season'] == yr)]
        if len(sub) == 0:
            continue
        latest = sub[sub['RankingDayNum'] == sub['RankingDayNum'].max()][['Season','TeamID','OrdinalRank']].copy()
        latest.rename(columns={'OrdinalRank': f'{system}_Rank'}, inplace=True)
        results.append(latest)
    return pd.concat(results)

kenpom = grab_rankings('POM')
print(f"KenPom rankings: {kenpom.shape}")

## Build Game Dataset

One row per tournament game going back to 2002. We always frame it as higher seed (better team) vs lower seed so our target is consistent — did the favorite win?

In [ ]:
# Strip the region letter from seeds so we just get the number
seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
seeds_clean = seeds[['Season','TeamID','SeedNum']].copy()

games = tourney[tourney['Season'] >= 2002].copy()

# Add seed numbers and team names for winner and loser
games = games.merge(seeds_clean.rename(columns={'TeamID':'WTeamID','SeedNum':'WSeed'}), on=['Season','WTeamID'], how='left')
games = games.merge(seeds_clean.rename(columns={'TeamID':'LTeamID','SeedNum':'LSeed'}), on=['Season','LTeamID'], how='left')
games = games.merge(teams[['TeamID','TeamName']].rename(columns={'TeamID':'WTeamID','TeamName':'WName'}), on='WTeamID', how='left')
games = games.merge(teams[['TeamID','TeamName']].rename(columns={'TeamID':'LTeamID','TeamName':'LName'}), on='LTeamID', how='left')

# Reframe as favorite vs underdog
# Lower seed number = better team (1 seed is better than 16)
games['FavWon']     = (games['WSeed'] <= games['LSeed']).astype(int)
games['FavID']      = np.where(games['WSeed'] <= games['LSeed'], games['WTeamID'], games['LTeamID'])
games['UnderdogID'] = np.where(games['WSeed'] <= games['LSeed'], games['LTeamID'], games['WTeamID'])
games['FavSeed']    = games[['WSeed','LSeed']].min(axis=1)
games['DogSeed']    = games[['WSeed','LSeed']].max(axis=1)
games['FavName']    = np.where(games['WSeed'] <= games['LSeed'], games['WName'], games['LName'])
games['DogName']    = np.where(games['WSeed'] <= games['LSeed'], games['LName'], games['WName'])

print(f"Games: {len(games)} | Years: {sorted(games['Season'].unique())}")
print(f"Favorite wins: {games['FavWon'].mean():.1%} of the time")
print(games[['Season','FavName','FavSeed','DogName','DogSeed','FavWon']].head(5))

## Merge Features

Attach efficiency stats to each game for both teams. The DEV dataset uses ESPN names while Mania uses its own naming so we need a mapping between them.

In [ ]:
# Stats we want from the DEV dataset
stat_cols = [
    'Adjusted Tempo', 'Raw Tempo', 'Adjusted Offensive Efficiency',
    'Raw Offensive Efficiency', 'Adjusted Defensive Efficiency',
    'Raw Defensive Efficiency', 'eFGPct', 'TOPct', 'ORPct', 'FTRate',
    'OffFT', 'Off2PtFG', 'Off3PtFG', 'DefFT', 'Def2PtFG', 'Def3PtFG',
    'Tempo', 'AdjTempo', 'OE', 'AdjOE', 'DE', 'AdjDE', 'AdjEM',
    'FG2Pct', 'FG3Pct', 'FTPct', 'BlockPct', 'OppFG2Pct', 'OppFG3Pct',
    'OppFTPct', 'OppBlockPct', 'FG3Rate', 'OppFG3Rate', 'ARate',
    'OppARate', 'StlRate', 'OppStlRate', 'Net Rating',
]
dev_stats = dev[['Season', 'Mapped ESPN Team Name'] + stat_cols].copy()
dev_stats = dev_stats.rename(columns={'Mapped ESPN Team Name': 'TeamName'})

# Team name differences between the two datasets
# Massey/Mania name -> DEV/ESPN name
name_fixes = {
    'Connecticut': 'UConn', 'Utah St': 'Utah State', 'Wright St': 'Wright State',
    'N Dakota St': 'North Dakota State', 'Colorado St': 'Colorado State',
    'Florida St': 'Florida State', 'Kansas St': 'Kansas State',
    'Oklahoma St': 'Oklahoma State', 'Oregon St': 'Oregon State',
    'Washington St': 'Washington State', 'Boise St': 'Boise State',
    'San Diego St': 'San Diego State', 'Arizona St': 'Arizona State',
    'Wichita St': 'Wichita State', 'FL Atlantic': 'Florida Atlantic',
    'FGCU': 'Florida Gulf Coast', 'F Dickinson': 'Fairleigh Dickinson',
    'G Washington': 'George Washington', 'Loyola-Chicago': 'Loyola Chicago',
    'Loyola MD': 'Loyola Maryland', 'Mississippi': 'Ole Miss',
    'Mississippi St': 'Mississippi State', 'Monmouth NJ': 'Monmouth',
    'Morehead St': 'Morehead State', "Mt St Mary's": "Mount St. Mary's",
    'Murray St': 'Murray State', 'NC A&T': 'North Carolina A&T',
    'Norfolk St': 'Norfolk State', 'S Illinois': 'Southern Illinois',
    'SUNY Albany': 'UAlbany', 'St Bonaventure': 'St. Bonaventure',
    "St Joseph's PA": "Saint Joseph's", 'TAM C. Christi': 'Texas A&M-Corpus Christi',
    'TX Southern': 'Texas Southern', 'UT San Antonio': 'UTSA',
    'WKU': 'Western Kentucky', 'Abilene Chr': 'Abilene Christian',
    'Alcorn St': 'Alcorn State', 'American Univ': 'American University',
    'Appalachian St': 'App State', 'Ark Little Rock': 'Little Rock',
    'Ark Pine Bluff': 'Arkansas Pine Bluff', 'Boston Univ': 'Boston University',
    'C Michigan': 'Central Michigan', 'CS Bakersfield': 'Cal State Bakersfield',
    'CS Fullerton': 'Cal State Fullerton', 'CS Northridge': 'Cal State Northridge',
    'Central Conn': 'Central Connecticut', 'Cleveland St': 'Cleveland State',
    'Coastal Car': 'Coastal Carolina', 'Col Charleston': 'Charleston',
    'Coppin St': 'Coppin State', 'Delaware St': 'Delaware State',
    'Detroit': 'Detroit Mercy', 'E Kentucky': 'Eastern Kentucky',
    'E Washington': 'Eastern Washington', 'ETSU': 'East Tennessee State',
    'Fresno St': 'Fresno State', 'Gardner Webb': 'Gardner-Webb',
    'Georgia St': 'Georgia State', 'IL Chicago': 'UIC',
    'Indiana St': 'Indiana State', 'Jackson St': 'Jackson State',
    'Jacksonville St': 'Jacksonville State', 'Kent': 'Kent State',
    'Kennesaw': 'Kennesaw State', 'LIU Brooklyn': 'LIU',
    'Long Beach St': 'Long Beach State', 'MS Valley St': 'Mississippi Valley State',
    'MTSU': 'Middle Tennessee', 'McNeese St': 'McNeese',
    'Miami OH': 'Miami (OH)', 'Montana St': 'Montana State',
    'N Colorado': 'Northern Colorado', 'N Kentucky': 'Northern Kentucky',
    'NC Central': 'North Carolina Central', 'NE Omaha': 'Omaha',
    'New Mexico St': 'New Mexico State', 'Northwestern LA': 'Northwestern State',
    'Penn St': 'Penn State', 'Portland St': 'Portland State',
    'S Carolina St': 'South Carolina State', 'S Dakota St': 'South Dakota State',
    'SE Missouri St': 'Southeast Missouri State', 'SF Austin': 'Stephen F. Austin',
    'SIUE': 'SIU Edwardsville', 'Sam Houston St': 'Sam Houston',
    'Southern Univ': 'Southern', 'St Francis PA': 'St. Francis (PA)',
    "St Peter's": "Saint Peter's", 'Alabama St': 'Alabama State',
    'W Michigan': 'Western Michigan', 'WI Green Bay': 'Green Bay',
    'WI Milwaukee': 'Milwaukee', 'Weber St': 'Weber State',
    'Penn': 'Pennsylvania', 'Prairie View': 'Prairie View A&M',
    'Queens NC': 'Queens University', "St John's": "St. John's",
    'St Louis': 'Saint Louis', "St Mary's CA": "Saint Mary's",
    'Tennessee St': 'Tennessee State', 'Michigan St': 'Michigan State',
    'Ohio St': 'Ohio State', 'Iowa St': 'Iowa State',
    'Miami FL': 'Miami', 'NC State': 'NC State',
    'Morgan St': 'Morgan State', 'Winthrop': 'Winthrop',
    'James Madison': 'James Madison', 'Radford': 'Radford',
}

# Flip it so we can map DEV names back to Massey names for the merge
reverse_map = {v: k for k, v in name_fixes.items()}
dev_stats['MasseyName'] = dev_stats['TeamName'].replace(reverse_map)
print(f"Stats table: {dev_stats.shape}")

In [ ]:
# Merge stats for the favorite
df = games.merge(
    dev_stats.rename(columns={'MasseyName':'FavName'}).add_suffix('_fav')
             .rename(columns={'Season_fav':'Season','FavName_fav':'FavName'}),
    on=['Season','FavName'], how='left'
)

# Merge stats for the underdog
df = df.merge(
    dev_stats.rename(columns={'MasseyName':'DogName'}).add_suffix('_dog')
             .rename(columns={'Season_dog':'Season','DogName_dog':'DogName'}),
    on=['Season','DogName'], how='left'
)

# Add win records and KenPom rankings
df = df.merge(record.rename(columns={'TeamID':'FavID','WinPct':'FavWinPct'}), on=['Season','FavID'], how='left')
df = df.merge(record.rename(columns={'TeamID':'UnderdogID','WinPct':'DogWinPct'}), on=['Season','UnderdogID'], how='left')
df = df.merge(kenpom.rename(columns={'TeamID':'FavID','POM_Rank':'FavPOM'}), on=['Season','FavID'], how='left')
df = df.merge(kenpom.rename(columns={'TeamID':'UnderdogID','POM_Rank':'DogPOM'}), on=['Season','UnderdogID'], how='left')

print(f"Merged: {df.shape}")
key_check = ['AdjEM_fav','AdjEM_dog','FavWinPct','DogWinPct','FavPOM','DogPOM']
print("Null % in key columns:")
print((df[key_check].isna().mean() * 100).round(1))

## Feature Engineering

For every stat we take favorite minus underdog — positive values mean the favorite is better. We also keep the individual seed numbers and win percentages as standalone features rather than just the difference, since a 1 vs 16 game is very different from a 5 vs 8 even if the efficiency gap is the same.

In [ ]:
# Drop games where we're missing stats (small schools not in DEV)
training_data = df.dropna(subset=['AdjEM_fav','AdjEM_dog']).copy()
print(f"Training games: {len(training_data)} (dropped {len(df) - len(training_data)})")

# Build difference features for each stat
diff_features = []
for col in stat_cols:
    fav_col = f'{col}_fav'
    dog_col = f'{col}_dog'
    if fav_col in training_data.columns and dog_col in training_data.columns:
        diff_name = f'{col}_diff'
        training_data[diff_name] = training_data[fav_col] - training_data[dog_col]
        diff_features.append(diff_name)

training_data['POM_diff'] = training_data['FavPOM'] - training_data['DogPOM']

# Keep individual values too, not just diffs
extra_features = ['FavSeed', 'DogSeed', 'FavWinPct', 'DogWinPct', 'FavPOM', 'DogPOM']
features = diff_features + ['POM_diff'] + extra_features

# Drop anything with too many nulls, fill the rest with median
null_pct = training_data[features].isna().mean()
features = [f for f in features if null_pct[f] < 0.10]
for col in features:
    training_data[col] = training_data[col].fillna(training_data[col].median())

print(f"Features: {len(features)} | Years: {sorted(training_data['Season'].unique())}")
print(f"Nulls remaining: {training_data[features].isna().sum().sum()}")

## Hyperparameter Tuning + Cross Validation

We use leave-one-year-out CV — standard k-fold would let the model train on 2023 data and test on 2015, which is data leakage. This way we always train on past years and test on future years.

Tuning is done with RandomizedSearchCV using the last 3 years as a holdout.

In [ ]:
X = training_data[features]
y = training_data['FavWon']
years = sorted(training_data['Season'].unique())

params = {
    'n_estimators':      [300, 500, 750, 1000],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

# Use 2023-2025 as validation set for tuning
val_mask = training_data['Season'].isin([2023, 2024, 2025])
split_idx = np.where(val_mask, 0, -1)
ps = PredefinedSplit(split_idx)

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    params, n_iter=40, scoring='neg_brier_score',
    cv=ps, random_state=42, n_jobs=-1, verbose=1
)
search.fit(X, y)
best_params = search.best_params_
print(f"Best params: {best_params}")
print(f"Tuning Brier: {-search.best_score_:.4f}")

In [ ]:
# Leave-one-year-out CV with the tuned params
cv_scores = []
correct = 0
total = 0

for yr in years:
    train = training_data['Season'] != yr
    test  = training_data['Season'] == yr
    model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    model.fit(X[train], y[train])
    probs = model.predict_proba(X[test])[:,1]
    preds = (probs >= 0.5).astype(int)
    cv_scores.append(brier_score_loss(y[test], probs))
    correct += (preds == y[test]).sum()
    total += test.sum()
    print(f"{yr}  Brier: {cv_scores[-1]:.4f}  ({test.sum()} games)")

print(f"\nMean Brier: {np.mean(cv_scores):.4f}")
print(f"Accuracy:   {correct/total:.1%} ({correct}/{total})")
print(f"Std:        {np.std(cv_scores):.4f}")
print(f"Best year:  {min(cv_scores):.4f}  |  Worst: {max(cv_scores):.4f}")
print(f"\n2024 competition winner: 0.1330  |  Ours: {np.mean(cv_scores):.4f}")

## Train Final Model

Train on all years (2002-2025) with the tuned params. No holdout here — we want every data point we can get.

In [ ]:
rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
rf.fit(X, y)
print(f"Trained on {len(training_data)} games ({min(years)}-{max(years)})")

# Feature importance
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(f"\nTop 15 features:")
print(importances.head(15))

fig, ax = plt.subplots(figsize=(8, 6))
importances.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

with open('rf_march_madness_2026.pkl', 'wb') as f:
    pickle.dump({'model': rf, 'features': features, 'params': best_params}, f)
print("Model saved.")

## Generate Predictions — All 2278 Matchups

The competition scores every game that actually gets played, regardless of what path we predicted. So we need a probability for every possible pairing of the 68 teams.

In [ ]:
# Get 2026 stats indexed by team name
dev_2026 = dev[dev['Season'] == 2026].copy().set_index('Mapped ESPN Team Name')

# Submission template uses slightly different names than DEV
sub_names = {
    'Iowa St': 'Iowa State', 'Michigan St': 'Michigan State',
    'Ohio St': 'Ohio State', 'Connecticut': 'UConn',
    "St John's": "St. John's", "St Mary's CA": "Saint Mary's",
    'St Louis': 'Saint Louis', 'NC State': 'NC State',
    'N Dakota St': 'North Dakota State', 'Prairie View': 'Prairie View A&M',
    'Queens NC': 'Queens University', 'Tennessee St': 'Tennessee State',
    'Utah St': 'Utah State', 'Wright St': 'Wright State',
    'Kennesaw': 'Kennesaw State', 'McNeese St': 'McNeese',
    'Miami FL': 'Miami', 'Miami OH': 'Miami (OH)',
    'LIU Brooklyn': 'LIU', 'Cal Baptist': 'California Baptist',
    'Penn': 'Pennsylvania', 'N Iowa': 'Northern Iowa',
    'SMU': 'SMU', 'TEX/NCST': 'NC State',
}

seeds_2026 = pd.read_csv(os.path.join(mania_path, 'MNCAATourneySeeds.csv'))
seeds_2026 = seeds_2026[seeds_2026['Season'] == 2026].merge(teams[['TeamID','TeamName']], on='TeamID')
team_ids_2026 = dict(zip(seeds_2026['TeamName'], seeds_2026['TeamID']))

def get_stats(name):
    dev_name = sub_names.get(name, name)
    if dev_name in dev_2026.index:
        return dev_2026.loc[dev_name]
    if name in dev_2026.index:
        return dev_2026.loc[name]
    return dev_2026.mean(numeric_only=True)

def get_winpct(name):
    tid = team_ids_2026.get(name)
    if tid is None:
        for k, v in name_fixes.items():
            if v == sub_names.get(name, name):
                tid = team_ids_2026.get(k)
                break
    if tid:
        row = record[(record['Season'] == 2026) & (record['TeamID'] == tid)]
        if len(row) > 0:
            return row.iloc[0]['WinPct']
    return 0.75

def get_pom(name):
    tid = team_ids_2026.get(name)
    if tid is None:
        for k, v in name_fixes.items():
            if v == sub_names.get(name, name):
                tid = team_ids_2026.get(k)
                break
    if tid:
        row = kenpom[(kenpom['Season'] == 2026) & (kenpom['TeamID'] == tid)]
        if len(row) > 0:
            return row.iloc[0]['POM_Rank']
    return 100

def predict_matchup(fav, fav_seed, dog, dog_seed):
    fav_stats = get_stats(fav)
    dog_stats = get_stats(dog)
    row = {}
    for col in stat_cols:
        diff_name = f'{col}_diff'
        if diff_name in features:
            row[diff_name] = fav_stats.get(col, 0) - dog_stats.get(col, 0)
    pom_fav = get_pom(fav)
    pom_dog = get_pom(dog)
    row['POM_diff']  = pom_fav - pom_dog
    row['FavPOM']    = pom_fav
    row['DogPOM']    = pom_dog
    row['FavSeed']   = fav_seed
    row['DogSeed']   = dog_seed
    row['FavWinPct'] = get_winpct(fav)
    row['DogWinPct'] = get_winpct(dog)
    return rf.predict_proba(pd.DataFrame([row])[features])[0][1]

# Run all 2278 predictions
preds = []
for _, row in matchups.iterrows():
    p = predict_matchup(row['HigherSeed'], row['HigherSeedNum'], row['LowerSeed'], row['LowerSeedNum'])
    preds.append(round(p, 4))

matchups['Predictions'] = preds
submission = matchups[['HigherSeed','LowerSeed','Predictions']]
submission.to_csv('submission_2026.csv', index=False)

print(f"Saved submission_2026.csv ({len(submission)} rows)")
print(f"Range: {submission['Predictions'].min():.3f} - {submission['Predictions'].max():.3f}")
print(f"Mean:  {submission['Predictions'].mean():.3f}")
print(f"\n1 vs 16 sanity check:")
print(matchups[(matchups['HigherSeedNum'] == 1) & (matchups['LowerSeedNum'] == 16)][['HigherSeed','LowerSeed','Predictions']].to_string())

## 2026 Bracket Simulation

Simulate the bracket round by round, advancing whoever the model favors each game.

In [ ]:
bracket = {
    'East': [
        (1,'Duke'),(16,'Siena'),(8,'Ohio St'),(9,'TCU'),
        (5,"St John's"),(12,'N Iowa'),(4,'Kansas'),(13,'Cal Baptist'),
        (6,'Louisville'),(11,'South Florida'),(3,'Michigan St'),(14,'N Dakota St'),
        (7,'UCLA'),(10,'UCF'),(2,'Connecticut'),(15,'Furman'),
    ],
    'South': [
        (1,'Florida'),(16,'Lehigh'),(8,'Clemson'),(9,'Iowa'),
        (5,'Vanderbilt'),(12,'McNeese St'),(4,'Nebraska'),(13,'Troy'),
        (6,'North Carolina'),(11,'VCU'),(3,'Illinois'),(14,'Penn'),
        (7,"St Mary's CA"),(10,'Texas A&M'),(2,'Houston'),(15,'Idaho'),
    ],
    'West': [
        (1,'Arizona'),(16,'LIU Brooklyn'),(8,'Villanova'),(9,'Utah St'),
        (5,'Wisconsin'),(12,'High Point'),(4,'Arkansas'),(13,'Hawaii'),
        (6,'BYU'),(11,'NC State'),(3,'Gonzaga'),(14,'Kennesaw'),
        (7,'Miami FL'),(10,'Missouri'),(2,'Purdue'),(15,'Queens NC'),
    ],
    'Midwest': [
        (1,'Michigan'),(16,'UMBC'),(8,'Georgia'),(9,'St Louis'),
        (5,'Texas Tech'),(12,'Akron'),(4,'Alabama'),(13,'Hofstra'),
        (6,'Tennessee'),(11,'SMU'),(3,'Virginia'),(14,'Wright St'),
        (7,'Kentucky'),(10,'Santa Clara'),(2,'Iowa St'),(15,'Tennessee St'),
    ],
}

round_labels = {64:'Round of 64', 32:'Round of 32', 16:'Sweet 16', 8:'Elite 8', 4:'Final Four', 2:'Championship'}
bracket_log = []

def play_game(team_a, seed_a, team_b, seed_b):
    if seed_a <= seed_b:
        fav, fs, dog, ds = team_a, seed_a, team_b, seed_b
    else:
        fav, fs, dog, ds = team_b, seed_b, team_a, seed_a
    p_fav = predict_matchup(fav, fs, dog, ds)
    p_a   = p_fav if seed_a <= seed_b else 1 - p_fav
    winner = team_a if p_a >= 0.5 else team_b
    w_seed = seed_a if winner == team_a else seed_b
    return p_a, winner, w_seed

def run_region(region, field):
    print(f"\n{region.upper()}")
    current = list(field)
    rnd = 64
    while len(current) > 1:
        next_round = []
        print(f"  {round_labels[rnd]}")
        for i in range(0, len(current), 2):
            s_a, t_a = current[i]
            s_b, t_b = current[i+1]
            p_a, winner, w_seed = play_game(t_a, s_a, t_b, s_b)
            p_w = p_a if winner == t_a else 1 - p_a
            print(f"    ({s_a}) {t_a:20s} vs ({s_b}) {t_b:20s}  ->  ({w_seed}) {winner} [{p_w:.1%}]")
            bracket_log.append({
                'region': region, 'round': round_labels[rnd],
                'team_a': t_a, 'seed_a': s_a, 'team_b': t_b, 'seed_b': s_b,
                'prob_a': round(p_a, 3), 'predicted_winner': winner, 'winner_seed': w_seed
            })
            next_round.append((w_seed, winner))
        current = next_round
        rnd //= 2
    return current[0]

final_four = []
for region, field in bracket.items():
    champ = run_region(region, field)
    final_four.append((champ[0], champ[1], region))
    print(f"  {region} Champion: ({champ[0]}) {champ[1]}\n")

# Final Four
print("FINAL FOUR")
ff = [(final_four[0], final_four[1]), (final_four[2], final_four[3])]
title_game = []
for (s_a, t_a, r_a), (s_b, t_b, r_b) in ff:
    p_a, winner, w_seed = play_game(t_a, s_a, t_b, s_b)
    p_w = p_a if winner == t_a else 1 - p_a
    print(f"  ({s_a}) {t_a} [{r_a}] vs ({s_b}) {t_b} [{r_b}]  ->  ({w_seed}) {winner} [{p_w:.1%}]")
    bracket_log.append({
        'region': f'{r_a}/{r_b}', 'round': 'Final Four',
        'team_a': t_a, 'seed_a': s_a, 'team_b': t_b, 'seed_b': s_b,
        'prob_a': round(p_a, 3), 'predicted_winner': winner, 'winner_seed': w_seed
    })
    title_game.append((w_seed, winner))

# Championship
print("\nNATIONAL CHAMPIONSHIP")
(s_a, t_a), (s_b, t_b) = title_game
p_a, winner, w_seed = play_game(t_a, s_a, t_b, s_b)
p_w = p_a if winner == t_a else 1 - p_a
print(f"  ({s_a}) {t_a} vs ({s_b}) {t_b}  ->  CHAMPION: ({w_seed}) {winner} [{p_w:.1%}]")
bracket_log.append({
    'region': 'National', 'round': 'Championship',
    'team_a': t_a, 'seed_a': s_a, 'team_b': t_b, 'seed_b': s_b,
    'prob_a': round(p_a, 3), 'predicted_winner': winner, 'winner_seed': w_seed
})

pd.DataFrame(bracket_log).to_csv('2026_bracket_predictions.csv', index=False)
print(f"\nSaved 2026_bracket_predictions.csv ({len(bracket_log)} games)")

## Results

In [ ]:
print(f"Model:         Random Forest (tuned)")
print(f"Params:        {best_params}")
print(f"Training data: 2002-2025 ({len(training_data)} games)")
print(f"Features:      {len(features)}")
print(f"Brier Score:   {np.mean(cv_scores):.4f}")
print(f"Accuracy:      {correct/total:.1%} ({correct}/{total} games)")
print(f"2024 winner:   0.1330")
print(f"Our model:     {np.mean(cv_scores):.4f}")
print(f"Prediction:    Duke over Arizona")